# 05 — Temporal Difference Learning

## Learning Objectives
1. Implement TD(0) prediction and track convergence via RMSE on random walk
2. Extend to TD(lambda) with eligibility traces; compare lambda values
3. Analyze online vs offline TD updates and n-step TD returns
4. Apply semi-gradient TD with linear function approximation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional

np.random.seed(42)

try:
    import torch
    import torch.nn as nn
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    device = 'cpu'

# --- Random Walk Environment ---
# 7 states: 0(terminal-left), 1-5(interior), 6(terminal-right=reward +1)
# Agent starts at state 3, moves left/right uniformly, episode ends at 0 or 6
# True V: V(s) = s/6 for s in {1,2,3,4,5}
TRUE_V = np.array([0.0, 1/6, 2/6, 3/6, 4/6, 5/6, 0.0])

print(f'numpy {np.__version__}, torch={TORCH_AVAILABLE}')
print(f'True V: {TRUE_V.round(4)}')
print('Random walk: 7 states (0=terminal-left, 6=terminal-right, reward=1)')


## Level 1: TD(0) Prediction on Random Walk

In [ ]:
def random_walk_episode(start: int = 3) -> List[Tuple[int, float, int]]:
    """Generate one random walk episode.

    Returns:
        List of (state, reward, next_state) tuples.
        Reward is 1.0 only on transition to state 6 (right terminal).
    """
    s = start
    trajectory = []
    while True:
        ns = s + np.random.choice([-1, 1])  # uniform left/right
        r = 1.0 if ns == 6 else 0.0
        trajectory.append((s, r, ns))
        s = ns
        if s == 0 or s == 6:  # terminal
            break
    return trajectory


def td0_prediction(
    n_episodes: int,
    alpha: float = 0.1,
    gamma: float = 1.0,
    v_init: float = 0.5,
    track_rmse: bool = True
) -> Tuple[np.ndarray, List[float]]:
    """TD(0) prediction via one-step bootstrapping.

    Update: V(s) <- V(s) + alpha * [r + gamma*V(s') - V(s)]

    Args:
        n_episodes: Number of training episodes.
        alpha: Learning rate (step size).
        gamma: Discount factor.
        v_init: Initial value estimate for non-terminal states.
        track_rmse: Whether to record RMSE after each episode.

    Returns:
        V: Value array shape (7,).
        rmse_history: RMSE vs TRUE_V per episode.
    """
    V = np.full(7, v_init)
    V[0] = V[6] = 0.0  # terminal states have value 0
    rmse_history = []

    for ep in range(n_episodes):
        trajectory = random_walk_episode()
        for s, r, ns in trajectory:
            if 0 < s < 6:  # only update non-terminal states
                td_error = r + gamma * V[ns] - V[s]
                V[s] += alpha * td_error
        if track_rmse:
            rmse = np.sqrt(np.mean((V[1:6] - TRUE_V[1:6])**2))
            rmse_history.append(rmse)

    return V, rmse_history


# Compare alpha values: 0.05, 0.1, 0.2, 0.5
alphas = [0.05, 0.1, 0.2, 0.5]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

alpha_results = {}
for a in alphas:
    np.random.seed(42)
    V_a, rmse_a = td0_prediction(n_episodes=200, alpha=a)
    alpha_results[a] = {'V': V_a, 'rmse': rmse_a}
    axes[0].plot(rmse_a, label=f'alpha={a}', linewidth=1.8, alpha=0.85)

axes[0].set_xlabel('Episode'); axes[0].set_ylabel('RMSE vs True V')
axes[0].set_title('TD(0) RMSE vs Episodes (different alpha)')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Best alpha: V profile
states = np.arange(1, 6)
axes[1].plot(states, TRUE_V[1:6], 'k--', label='True V', linewidth=2)
for a, color in zip([0.1, 0.2], ['steelblue', 'darkorange']):
    axes[1].plot(states, alpha_results[a]['V'][1:6], 'o-', color=color,
                 label=f'TD(0) alpha={a}', linewidth=1.8)
axes[1].set_xlabel('State'); axes[1].set_ylabel('V(s)')
axes[1].set_title('TD(0) V Estimates after 200 Episodes')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_05_td0_convergence.png', dpi=80, bbox_inches='tight')
plt.show()

print('TD(0) Results after 200 episodes:')
print(f'  {"Alpha":>7} {"Final RMSE":>12} {"V(s=3)":>10}')
for a in alphas:
    print(f'  {a:>7.2f} {alpha_results[a]["rmse"][-1]:>12.4f} '
          f'{alpha_results[a]["V"][3]:>10.4f}')
print(f'  True V(s=3): {TRUE_V[3]:.4f}')


## Level 2: TD(lambda) with Eligibility Traces

In [ ]:
def td_lambda_prediction(
    n_episodes: int,
    lam: float,
    alpha: float = 0.1,
    gamma: float = 1.0,
    v_init: float = 0.5
) -> Tuple[np.ndarray, List[float]]:
    """TD(lambda) prediction with accumulating eligibility traces.

    Eligibility traces: e(s) <- gamma*lambda*e(s) + 1 (for visited s)
    Update: V(s) <- V(s) + alpha * delta * e(s) for all s
    where delta = r + gamma*V(s') - V(s) is the TD error.

    lambda=0 -> TD(0), lambda=1 -> MC (in episodic tasks).

    Args:
        lam: Trace decay parameter in [0, 1].
        alpha: Learning rate.
        gamma: Discount factor.

    Returns:
        V: Value estimates after n_episodes.
        rmse_history: RMSE vs TRUE_V per episode.
    """
    V = np.full(7, v_init)
    V[0] = V[6] = 0.0
    rmse_history = []

    for ep in range(n_episodes):
        e = np.zeros(7)  # eligibility trace (reset each episode)
        trajectory = random_walk_episode()

        for s, r, ns in trajectory:
            # Accumulate trace for visited state
            e[s] += 1.0

            # TD error: how much V(s) is wrong given observed (r, V(s'))
            delta = r + gamma * V[ns] - V[s]

            # Update ALL states proportional to their eligibility
            V += alpha * delta * e
            V[0] = V[6] = 0.0  # clamp terminals

            # Decay traces
            e *= gamma * lam

        rmse = np.sqrt(np.mean((V[1:6] - TRUE_V[1:6])**2))
        rmse_history.append(rmse)

    return V, rmse_history


# Sweep lambda values
lambdas = [0.0, 0.3, 0.5, 0.9, 1.0]
lambda_colors = ['steelblue', 'darkorange', 'green', 'red', 'purple']
lambda_results = {}

print('Running TD(lambda) for lambdas:', lambdas)
for lam in lambdas:
    np.random.seed(42)
    V_lam, rmse_lam = td_lambda_prediction(n_episodes=200, lam=lam, alpha=0.1)
    lambda_results[lam] = {'V': V_lam, 'rmse': rmse_lam}
    print(f'  lambda={lam:.1f}: final RMSE={rmse_lam[-1]:.4f}, V(3)={V_lam[3]:.4f}')

fig_lam, axes_lam = plt.subplots(1, 2, figsize=(14, 5))

for lam, color in zip(lambdas, lambda_colors):
    axes_lam[0].plot(lambda_results[lam]['rmse'],
                     label=f'lambda={lam}', color=color, linewidth=1.8, alpha=0.85)
axes_lam[0].set_xlabel('Episode'); axes_lam[0].set_ylabel('RMSE')
axes_lam[0].set_title('TD(lambda): RMSE vs Episodes')
axes_lam[0].legend(fontsize=9); axes_lam[0].grid(True, alpha=0.3)

# Final RMSE vs lambda (U-shaped: best in the middle)
final_rmses = [lambda_results[lam]['rmse'][-1] for lam in lambdas]
axes_lam[1].plot(lambdas, final_rmses, 'o-', color='navy', linewidth=2, markersize=8)
best_lam = lambdas[np.argmin(final_rmses)]
axes_lam[1].axvline(best_lam, color='red', linestyle='--', alpha=0.6, label=f'Best lambda={best_lam}')
axes_lam[1].set_xlabel('Lambda'); axes_lam[1].set_ylabel('Final RMSE (after 200 ep)')
axes_lam[1].set_title('Final RMSE vs Lambda (U-shaped curve)')
axes_lam[1].legend(); axes_lam[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_05_td_lambda.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'Best lambda: {best_lam} (minimizes RMSE after 200 episodes)')
print('U-shaped curve: lambda=0 (TD bias), lambda=1 (MC variance), best is in-between.')


## Real-World Example 1: Online vs Offline TD Updates

In [ ]:
# Online TD: update V immediately after each step within the episode
# Offline TD: collect full episode, then batch update at the end
# This matters for TD(lambda) convergence behavior

def offline_td_lambda(
    n_episodes: int,
    lam: float,
    alpha: float = 0.05,
    gamma: float = 1.0
) -> Tuple[np.ndarray, List[float]]:
    """Offline TD(lambda): accumulate full episode, then compute lambda-return and update.

    G_t^lambda = (1-lambda)*sum_{n=1}^{T-t-1} lambda^{n-1} * G_t^{(n)} + lambda^{T-t-1} * G_t
    This is the 'forward view' of TD(lambda) (offline version).
    """
    V = np.full(7, 0.5)
    V[0] = V[6] = 0.0
    rmse_history = []

    for ep in range(n_episodes):
        traj = random_walk_episode()
        T = len(traj)
        states = [s for s, r, ns in traj]
        rewards = [r for s, r, ns in traj]

        # Compute n-step returns G_t^(n) for each step t
        n_step_returns = np.zeros((T, T))  # n_step_returns[t, n] = G_t^(n+1)
        for t in range(T):
            G = 0.0
            for n in range(T - t):
                G = G + (gamma ** n) * rewards[t + n]
                n_step_returns[t, n] = G + (gamma ** (n + 1)) * V[traj[t + n][2]]
            # Full Monte Carlo return from t
            n_step_returns[t, T - t - 1] = sum(gamma**k * rewards[t+k] for k in range(T-t))

        # Lambda-return: G_t^lambda = (1-lam)*sum + lam^(T-t-1)*MC
        updates = np.zeros(7)
        update_counts = np.zeros(7)
        for t in range(T):
            s = states[t]
            if s == 0 or s == 6:
                continue
            G_lambda = 0.0
            n_steps = T - t
            for n in range(n_steps - 1):
                G_lambda += (1 - lam) * (lam ** n) * n_step_returns[t, n]
            G_lambda += (lam ** (n_steps - 1)) * n_step_returns[t, n_steps - 1]
            updates[s] += alpha * (G_lambda - V[s])
            update_counts[s] += 1

        # Batch update after episode
        mask = update_counts > 0
        V[mask] += updates[mask] / update_counts[mask]
        V[0] = V[6] = 0.0

        rmse = np.sqrt(np.mean((V[1:6] - TRUE_V[1:6])**2))
        rmse_history.append(rmse)

    return V, rmse_history


print('Comparing online vs offline TD(lambda) at lambda=0.5...')
np.random.seed(42)
V_online, rmse_online = td_lambda_prediction(n_episodes=200, lam=0.5, alpha=0.1)
np.random.seed(42)
V_offline, rmse_offline = offline_td_lambda(n_episodes=200, lam=0.5, alpha=0.1)

fig_on_off, axes_on_off = plt.subplots(1, 2, figsize=(13, 5))
axes_on_off[0].plot(rmse_online, label='Online TD(lambda=0.5)', color='steelblue', linewidth=2)
axes_on_off[0].plot(rmse_offline, label='Offline TD(lambda=0.5)', color='darkorange', linewidth=2, linestyle='--')
axes_on_off[0].set_xlabel('Episode'); axes_on_off[0].set_ylabel('RMSE')
axes_on_off[0].set_title('Online vs Offline TD(lambda=0.5)')
axes_on_off[0].legend(); axes_on_off[0].grid(True, alpha=0.3)

states_p = np.arange(1, 6)
axes_on_off[1].plot(states_p, TRUE_V[1:6], 'k--', label='True V', linewidth=2)
axes_on_off[1].plot(states_p, V_online[1:6], 'o-', color='steelblue', label='Online', linewidth=2)
axes_on_off[1].plot(states_p, V_offline[1:6], 's--', color='darkorange', label='Offline', linewidth=2)
axes_on_off[1].set_xlabel('State'); axes_on_off[1].set_ylabel('V(s)')
axes_on_off[1].set_title('V Estimates: Online vs Offline')
axes_on_off[1].legend(); axes_on_off[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_05_online_offline.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'Online  RMSE after 200 ep: {rmse_online[-1]:.4f}, V(3)={V_online[3]:.4f}')
print(f'Offline RMSE after 200 ep: {rmse_offline[-1]:.4f}, V(3)={V_offline[3]:.4f}')
print('Online updates faster per step; offline is more stable (batch update).')


## Real-World Example 2: n-Step TD Returns

In [ ]:
# n-step TD: G_t^(n) = r_t + gamma*r_{t+1} + ... + gamma^{n-1}*r_{t+n-1} + gamma^n*V(s_{t+n})
# n=1 -> TD(0), n=infinity -> MC. Shows bias-variance tradeoff.

def n_step_td_prediction(
    n_step: int,
    n_episodes: int,
    alpha: float = 0.1,
    gamma: float = 1.0
) -> Tuple[np.ndarray, List[float]]:
    """n-step TD prediction for random walk.

    Uses n-step return G_t^(n) as the update target instead of 1-step TD or full MC.
    """
    V = np.full(7, 0.5)
    V[0] = V[6] = 0.0
    rmse_history = []

    for ep in range(n_episodes):
        traj = random_walk_episode()
        T = len(traj)
        states = [s for s, r, ns in traj] + [traj[-1][2]]  # include final next_state
        rewards = [r for s, r, ns in traj]

        for t in range(T):
            s = states[t]
            if s == 0 or s == 6:
                continue
            # n-step return from step t
            n_end = min(t + n_step, T)
            G_n = sum(gamma**(k - t) * rewards[k] for k in range(t, n_end))
            if n_end < len(states):
                G_n += gamma**n_step * V[states[n_end]]
            td_err = G_n - V[s]
            V[s] += alpha * td_err
        V[0] = V[6] = 0.0

        rmse = np.sqrt(np.mean((V[1:6] - TRUE_V[1:6])**2))
        rmse_history.append(rmse)

    return V, rmse_history


n_steps_list = [1, 2, 4, 8, 'inf']
n_step_colors = ['steelblue', 'darkorange', 'green', 'red', 'purple']
n_step_results = {}

print('n-step TD on random walk (alpha=0.1, 200 episodes):')
for n_s in n_steps_list:
    np.random.seed(42)
    if n_s == 'inf':  # MC prediction
        rs = np.zeros(7); rc = np.zeros(7)
        rmselist = []
        V_mc_rw = np.full(7, 0.5); V_mc_rw[0] = V_mc_rw[6] = 0.0
        for _ in range(200):
            traj = random_walk_episode()
            G = 0.0; vis = set()
            for s, r, ns in reversed(traj):
                G = G + r
                if s not in vis:
                    vis.add(s); rs[s] += G; rc[s] += 1
            V_mc_rw = np.where(rc > 0, rs/rc, V_mc_rw)
            V_mc_rw[0] = V_mc_rw[6] = 0.0
            rmselist.append(np.sqrt(np.mean((V_mc_rw[1:6] - TRUE_V[1:6])**2)))
        n_step_results[n_s] = {'V': V_mc_rw, 'rmse': rmselist}
    else:
        V_ns, rmse_ns = n_step_td_prediction(n_step=n_s, n_episodes=200, alpha=0.1)
        n_step_results[n_s] = {'V': V_ns, 'rmse': rmse_ns}
    fin = n_step_results[n_s]['rmse'][-1]
    v3 = n_step_results[n_s]['V'][3]
    print(f'  n={str(n_s):>4}: final RMSE={fin:.4f}, V(3)={v3:.4f}')

fig_nstep, axes_ns = plt.subplots(1, 2, figsize=(14, 5))
for n_s, color in zip(n_steps_list, n_step_colors):
    axes_ns[0].plot(n_step_results[n_s]['rmse'],
                    label=f'n={n_s}', color=color, linewidth=1.8, alpha=0.85)
axes_ns[0].set_xlabel('Episode'); axes_ns[0].set_ylabel('RMSE')
axes_ns[0].set_title('n-step TD: RMSE vs Episodes')
axes_ns[0].legend(); axes_ns[0].grid(True, alpha=0.3)

final_rmse_n = [n_step_results[n_s]['rmse'][-1] for n_s in n_steps_list]
x_labels = [str(n) for n in n_steps_list]
axes_ns[1].bar(x_labels, final_rmse_n,
               color=n_step_colors[:len(n_steps_list)], alpha=0.8)
axes_ns[1].set_xlabel('n (step horizon)'); axes_ns[1].set_ylabel('Final RMSE')
axes_ns[1].set_title('Bias-Variance Tradeoff: Final RMSE vs n')
axes_ns[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('rl_05_nstep_td.png', dpi=80, bbox_inches='tight')
plt.show()

print('n=1 (TD0): high bias (bootstraps from imprecise V); n=inf (MC): high variance.')
print('Optimal n typically 4-8 for this random walk (minimizes RMSE).')


## Real-World Example 3: TD in Function Approximation (Semi-Gradient TD)

In [ ]:
# Linear value function approximation: V(s; w) = w @ phi(s)
# phi(s): one-hot feature vector for state s (tabular as linear FA)
# Semi-gradient TD: w <- w + alpha * delta * grad_w V(s)
# = w + alpha * [r + gamma*V(s';w) - V(s;w)] * phi(s)

def phi_onehot(s: int, n_states: int = 7) -> np.ndarray:
    """One-hot feature vector for state s."""
    vec = np.zeros(n_states)
    vec[s] = 1.0
    return vec


def phi_polynomial(s: int, degree: int = 3, n_states: int = 7) -> np.ndarray:
    """Polynomial features: [1, s/n, (s/n)^2, ..., (s/n)^degree]."""
    s_norm = s / (n_states - 1)  # normalize to [0, 1]
    return np.array([s_norm ** d for d in range(degree + 1)])


def semi_gradient_td0(
    feature_fn,
    n_features: int,
    n_episodes: int,
    alpha: float = 0.05,
    gamma: float = 1.0
) -> Tuple[np.ndarray, List[float], np.ndarray]:
    """Semi-gradient TD(0) with linear function approximation.

    V(s; w) = w^T phi(s)
    w update: w += alpha * [r + gamma*V(s';w) - V(s;w)] * phi(s)
    'Semi-gradient' because we stop gradient at the bootstrap target.
    """
    w = np.zeros(n_features)  # weight vector
    rmse_history = []

    for ep in range(n_episodes):
        traj = random_walk_episode()
        for s, r, ns in traj:
            if s == 0 or s == 6:
                continue
            phi_s = feature_fn(s)
            phi_ns = feature_fn(ns)
            V_s = w @ phi_s
            V_ns = w @ phi_ns
            td_error = r + gamma * V_ns - V_s
            w += alpha * td_error * phi_s  # semi-gradient update

        # Compute RMSE
        V_approx = np.array([w @ feature_fn(s) for s in range(7)])
        rmse = np.sqrt(np.mean((V_approx[1:6] - TRUE_V[1:6])**2))
        rmse_history.append(rmse)

    V_final = np.array([w @ feature_fn(s) for s in range(7)])
    return w, rmse_history, V_final


# Compare: one-hot (tabular) vs polynomial features
print('Semi-gradient TD(0): one-hot vs polynomial features...')
np.random.seed(42)
w_oh, rmse_oh, V_oh = semi_gradient_td0(
    lambda s: phi_onehot(s), n_features=7, n_episodes=500, alpha=0.05)
np.random.seed(42)
w_poly, rmse_poly, V_poly = semi_gradient_td0(
    lambda s: phi_polynomial(s, degree=3), n_features=4, n_episodes=500, alpha=0.1)

fig_fa, axes_fa = plt.subplots(1, 2, figsize=(13, 5))
axes_fa[0].plot(rmse_oh, label='One-hot (tabular)', color='steelblue', linewidth=2)
axes_fa[0].plot(rmse_poly, label='Polynomial degree-3', color='darkorange', linewidth=2)
axes_fa[0].set_xlabel('Episode'); axes_fa[0].set_ylabel('RMSE')
axes_fa[0].set_title('Semi-Gradient TD(0): Feature Comparison')
axes_fa[0].legend(); axes_fa[0].grid(True, alpha=0.3)

states_fa = np.arange(1, 6)
axes_fa[1].plot(states_fa, TRUE_V[1:6], 'k--', label='True V', linewidth=2)
axes_fa[1].plot(states_fa, V_oh[1:6], 'o-', color='steelblue', label='One-hot TD', linewidth=2)
axes_fa[1].plot(states_fa, V_poly[1:6], 's--', color='darkorange', label='Poly TD', linewidth=2)
axes_fa[1].set_xlabel('State'); axes_fa[1].set_ylabel('V(s)')
axes_fa[1].set_title('Approximated V vs True V')
axes_fa[1].legend(); axes_fa[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_05_fa_td.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'One-hot final RMSE:  {rmse_oh[-1]:.4f}  (same as tabular TD)')
print(f'Polynomial final RMSE: {rmse_poly[-1]:.4f}  (compact: 4 weights for 5 states)')
print('Polynomial FA can generalize across states; one-hot is exact but not compact.')


## Comparison: TD(lambda) at Different Lambda Values

In [ ]:
# Systematic comparison: run 10 trials per lambda, compute mean RMSE trajectory

n_trials_cmp = 10
n_ep_cmp = 200
lambdas_cmp = [0.0, 0.3, 0.5, 0.9, 1.0]
colors_cmp = ['steelblue', 'darkorange', 'green', 'red', 'purple']

print(f'Running {n_trials_cmp} trials per lambda ({n_ep_cmp} episodes each)...')
lambda_trial_rmse = {lam: [] for lam in lambdas_cmp}

for lam in lambdas_cmp:
    for trial in range(n_trials_cmp):
        np.random.seed(trial * 17 + 1)
        _, rmse_t = td_lambda_prediction(n_episodes=n_ep_cmp, lam=lam, alpha=0.1)
        lambda_trial_rmse[lam].append(rmse_t)

fig_cmp, axes_cmp = plt.subplots(1, 2, figsize=(14, 5))

# Mean RMSE trajectory per lambda
for lam, color in zip(lambdas_cmp, colors_cmp):
    rmse_mat = np.array(lambda_trial_rmse[lam])  # (n_trials, n_episodes)
    mean_rmse = rmse_mat.mean(axis=0)
    std_rmse = rmse_mat.std(axis=0)
    axes_cmp[0].plot(mean_rmse, label=f'lam={lam}', color=color, linewidth=2)
    axes_cmp[0].fill_between(np.arange(n_ep_cmp),
                             mean_rmse - std_rmse, mean_rmse + std_rmse,
                             alpha=0.15, color=color)
axes_cmp[0].set_xlabel('Episode'); axes_cmp[0].set_ylabel('Mean RMSE (+/- std)')
axes_cmp[0].set_title(f'TD(lambda) RMSE: Mean over {n_trials_cmp} Trials')
axes_cmp[0].legend(fontsize=9); axes_cmp[0].grid(True, alpha=0.3)

# Final RMSE vs lambda with error bars
final_means = [np.mean([lambda_trial_rmse[lam][t][-1] for t in range(n_trials_cmp)])
               for lam in lambdas_cmp]
final_stds = [np.std([lambda_trial_rmse[lam][t][-1] for t in range(n_trials_cmp)])
              for lam in lambdas_cmp]
axes_cmp[1].errorbar(lambdas_cmp, final_means, yerr=final_stds,
                     fmt='o-', color='navy', linewidth=2, capsize=6, markersize=8)
best_lam_idx = np.argmin(final_means)
axes_cmp[1].axvline(lambdas_cmp[best_lam_idx], color='red', linestyle='--', alpha=0.7,
                    label=f'Best: lambda={lambdas_cmp[best_lam_idx]}')
axes_cmp[1].set_xlabel('Lambda'); axes_cmp[1].set_ylabel('Final Mean RMSE')
axes_cmp[1].set_title(f'Final RMSE vs Lambda ({n_trials_cmp} trials, {n_ep_cmp} ep)')
axes_cmp[1].legend(); axes_cmp[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_05_lambda_comparison.png', dpi=80, bbox_inches='tight')
plt.show()

print('Summary table: Lambda -> mean final RMSE +/- std:')
for lam, m, s in zip(lambdas_cmp, final_means, final_stds):
    star = ' <-- BEST' if lam == lambdas_cmp[best_lam_idx] else ''
    print(f'  lambda={lam:.1f}: {m:.4f} +/- {s:.4f}{star}')

# n-step vs lambda comparison: show they encode similar bias-variance tradeoff
print('\nConceptual equivalence:')
print('  TD(lambda=0) ≈ n-step TD with n=1 (one-step bootstrap)')
print('  TD(lambda=1) ≈ n-step TD with n=inf (full MC return)')
print('  TD(lambda) is an exponential average over all n-step returns.')


## Key Takeaways

**Core idea:** TD learning bootstraps — it updates V(s) toward an estimated return r + gamma*V(s'), not waiting for the episode to end. This enables online learning on continuing tasks and reduces variance vs MC at the cost of bias.

| Method | Bias | Variance | Online | Function Approx |
|--------|------|----------|--------|-----------------|
| TD(0) | High (bootstrap) | Low | Yes | Semi-gradient |
| TD(lambda=0.5) | Medium | Medium | Yes | Semi-gradient |
| TD(lambda=1) = MC | None | High | No (needs episode) | Semi-gradient |
| n-step TD (n=4) | Medium | Low-Medium | Yes | Yes |

**Failure modes:**
- alpha too high: TD updates oscillate, RMSE diverges
- lambda close to 1 with function approximation: divergence risk (deadly triad)
- Online TD with non-i.i.d. data: temporal correlation slows learning

**Related:** [04-mc](04-monte-carlo-methods.ipynb), [06-q-learning](06-q-learning.ipynb)

## Exercises

1. **Alpha decay:** Implement decaying alpha = alpha_0 / (1 + t). Does decaying alpha improve final RMSE vs fixed alpha for TD(0)?
2. **Replacing traces:** Implement replacing eligibility traces: e(s) = 1 on visit (not accumulating). Compare to accumulating traces at lambda=0.9.
3. **Non-linear FA:** Replace linear FA with a 2-layer MLP (using torch). Observe if semi-gradient TD still converges.
4. **Larger random walk:** Extend to 19-state random walk (Sutton & Barto Table 9.2). Sweep alpha and lambda to find optimal combination.